# FEATURE ENGINEERING

In [21]:
import pandas as pd
import numpy as np
import seaborn as sns
import os
import ast

In this file we will do the following:

1. Load the data + set index (sort)
2. Create the target variable
3. Feature engineering (create the features)
4. Visualization

### LOAD THE DATA AND SET INDEX

We are first going to create a final dataset with all the information. We start by loading all the clean datasets, and checking the information in them.

In [22]:
# ACLED
acled = pd.read_csv("../data_clean/acled_clean.csv")
acled = acled.set_index(['iso3', 'month']).sort_index() 

list_cols = [
    "event_type",
    "sub_event_type",
    "disorder_type",
]

for col in list_cols:
    acled[col] = acled[col].apply(
        lambda x: ast.literal_eval(x) if isinstance(x, str) else x
    )

# IDMC
idmc = pd.read_csv("../data_clean/idmc_clean.csv")
idmc = idmc.set_index(['iso3', 'month']).sort_index()

# HDX
hdx = pd.read_csv("../data_clean/hdx_clean.csv")
hdx = hdx.set_index(['iso3', 'month']).sort_index()

hdx["hdx_alert_level"] = hdx["hdx_alert_level"].apply(
        lambda x: ast.literal_eval(x) if isinstance(x, str) else x
    )

# ECONAI
econAI = pd.read_csv("../data_clean/econAI_clean.csv")
econAI = econAI.set_index(['iso3', 'month']).sort_index()

# GOOGLE TRENDS
google = pd.read_csv("../data_clean/google_trends_clean.csv")
google = google.set_index(['iso3', 'month']).sort_index()

# INFORM INDEX
inform = pd.read_csv("../data_clean/inform_clean.csv")
inform = inform.set_index(['iso3', 'month']).sort_index()

In [23]:
datasets = {
    "IDMC": idmc,
    "ACLED": acled,
    "HDX": hdx,
    "EconAI": econAI,
    "Google Trends": google,
    "INFORM Index": inform
}

def analyze_datasets(datasets_dict):
    analysis = []
    
    for name, df in datasets_dict.items():
       
        temp_df = df.copy()
        temp_df = temp_df.reset_index()
        temp_df['month'] = pd.to_datetime(temp_df['month'])
        
        analysis.append({
            "Dataset": name,
            "Rows": len(temp_df),
            "Countries": temp_df['iso3'].nunique(),
            "Min_Date": temp_df['month'].min().strftime('%Y-%m'),
            "Max_Date": temp_df['month'].max().strftime('%Y-%m'),
            "Avg_Months": round(len(temp_df) / temp_df['iso3'].nunique(), 1)
        })
    
    return pd.DataFrame(analysis)

# Torna a executar-ho amb els teus dataframes
df_info = analyze_datasets(datasets)
print(df_info.to_string(index=False))

      Dataset  Rows  Countries Min_Date Max_Date  Avg_Months
         IDMC  8844         89  2018-01  2026-04        99.4
        ACLED 25236        238  1996-12  2026-03       106.0
          HDX  1209        107  1998-05  2026-03        11.3
       EconAI 35472        182  2010-01  2026-03       194.9
Google Trends  8888         88  2017-12  2026-04       101.0
 INFORM Index 19100        191  2018-01  2026-04       100.0


In [24]:
def check_temporal_gaps(df):
    df = df.copy().reset_index()
    df['month'] = pd.to_datetime(df['month'])
    df = df.sort_values(['iso3', 'month'])
    
    df['diff'] = df.groupby('iso3')['month'].diff() / pd.Timedelta(days=31)
    
    gaps = df[df['diff'] > 1.1]
    
    if gaps.empty:
        print("No temporal gaps found in the dataset.")
    else:
        print(f"{len(gaps)} temporal gaps found:")
        print(gaps[['iso3', 'month', 'diff']].head(10))
    return gaps

gaps_df = check_temporal_gaps(idmc)
gaps_df = check_temporal_gaps(acled)
gaps_df = check_temporal_gaps(hdx)
gaps_df = check_temporal_gaps(econAI)
gaps_df = check_temporal_gaps(google)
gaps_df = check_temporal_gaps(inform)

No temporal gaps found in the dataset.
2512 temporal gaps found:
   iso3      month      diff
1   ABW 2018-09-01  5.935484
2   ABW 2019-02-01  4.935484
4   ABW 2019-06-01  2.967742
8   ABW 2020-03-01  5.870968
9   ABW 2020-05-01  1.967742
11  ABW 2020-08-01  1.967742
12  ABW 2020-10-01  1.967742
14  ABW 2021-01-01  1.967742
15  ABW 2021-03-01  1.903226
16  ABW 2021-09-01  5.935484
965 temporal gaps found:
   iso3      month       diff
2   AFG 2018-02-01  10.870968
3   AFG 2018-05-01   2.870968
5   AFG 2018-09-01   2.967742
6   AFG 2019-03-01   5.838710
7   AFG 2019-09-01   5.935484
8   AFG 2020-04-01   6.870968
9   AFG 2020-08-01   3.935484
11  AFG 2021-04-01   6.838710
14  AFG 2021-10-01   3.935484
16  AFG 2022-06-01   6.838710
No temporal gaps found in the dataset.
No temporal gaps found in the dataset.
No temporal gaps found in the dataset.


Not a surprise that both ACLED and HDX Signals have temporal lags, as they only contain information for the date ans country where an event or alarm occur. So it's not a problem to have them.

So, regarning the missing data:

- ACLED: Missing values in the period covered by ACLED means that nothing happened there, so can be imputed (even a hole country).
- IDMC: Missing values for an entire country during the period covered by IDMC means that there hasen't been any displacement in that country, so we can impute them easily. We can't impute a gap but yes a hole country.
- HDX Signals: Same as ACLED.
- ECONAI: We can't impute anything.
- GOOGLE TRENDS: We can't impute anything
- INFORM Index: We can't impute anything

Based on that our dataset should contain only the countries contained in the intersection on ECONAI, GOOGLE trends and INFORM index. As Google trends we have scraped manually we just have the countries needed (the ones in thet intersection of EconAI and inform, during the periods covered by both of them), we don't have to worry about this one. Regarding the time periods, we need to keep only the periods covered by the intersection on IDMC, ECONAI, Google Trends and Inform index, that in this case is from 2018-01 to 2026-03.

In [25]:
# 1. Get the strict intersection of countries present in BOTH EconAI and INFORM
econAI_countries = set(econAI.reset_index()['iso3'].unique())
inform_countries = set(inform.index.get_level_values('iso3').unique())

# The target list: countries that MUST be in both datasets
target_countries = econAI_countries.intersection(inform_countries)

print("=" * 70)
print(f"🎯 TARGET COUNTRIES (Intersection of EconAI & INFORM): {len(target_countries)}")
print("=" * 70)

# 2. Check how many countries from each dataset are discarded/lost based on this target
for name, df_actual in datasets.items():
    actual_countries = set(df_actual.reset_index()['iso3'].unique())
    
    # Countries that are in the current dataset but WILL BE LOST 
    # because they are not in our target intersection
    discarded = actual_countries - target_countries
    
    # Countries that this dataset lacks to reach the target (if any)
    missing_from_target = target_countries - actual_countries
    
    print(f"📊 {name}:")
    print(f"   • Total countries in raw file: {len(actual_countries)}")
    print(f"   • Kept for analysis: {len(actual_countries.intersection(target_countries))}")
    
    if len(discarded) > 0:
        print(f"   ❌ Discarded countries (not in intersection): {len(discarded)} {sorted(list(discarded))}")
    else:
        print("   ✅ Perfect! No countries discarded from this dataset.")
        
    if len(missing_from_target) > 0:
        print(f"   ⚠️ Lacks these target countries (will cause NaNs): {sorted(list(missing_from_target))}")
        
    print("-" * 70)

🎯 TARGET COUNTRIES (Intersection of EconAI & INFORM): 176
📊 IDMC:
   • Total countries in raw file: 89
   • Kept for analysis: 86
   ❌ Discarded countries (not in intersection): 3 ['AB9', 'MYT', 'NCL']
   ⚠️ Lacks these target countries (will cause NaNs): ['ALB', 'ARE', 'ARG', 'AUT', 'BEL', 'BGR', 'BHS', 'BLZ', 'BRB', 'BRN', 'BTN', 'BWA', 'CAN', 'CHE', 'CHL', 'CHN', 'CRI', 'CUB', 'CZE', 'DEU', 'DNK', 'DOM', 'DZA', 'ERI', 'ESP', 'EST', 'FIN', 'FJI', 'GAB', 'GEO', 'GNB', 'GNQ', 'GRD', 'GTM', 'GUY', 'HRV', 'HUN', 'IRL', 'ISL', 'JAM', 'JOR', 'JPN', 'KOR', 'KWT', 'LAO', 'LSO', 'LTU', 'LUX', 'LVA', 'MAR', 'MDA', 'MDV', 'MKD', 'MLT', 'MNE', 'MNG', 'MRT', 'MUS', 'MYS', 'NAM', 'NOR', 'NPL', 'NZL', 'OMN', 'PAN', 'POL', 'PRK', 'PRT', 'PRY', 'RWA', 'SAU', 'SEN', 'SGP', 'SRB', 'STP', 'SVK', 'SVN', 'SWE', 'SWZ', 'SYC', 'TKM', 'TLS', 'TON', 'TTO', 'TUN', 'URY', 'UZB', 'VNM', 'VUT', 'WSM']
----------------------------------------------------------------------
📊 ACLED:
   • Total countries in raw file:

Let's do now the final dataset containing the countries in the intersecction of EconAI and INFORM, and the time periods in the intersection of EconAI, Inform and IDMC that is the same as the intersection of EconAI and INFORM:


In [32]:
df = econAI.join(inform, how="inner")

df = df.join(acled, how="left") \
        .join(hdx, how="left") \
        .join(idmc, how="left") \
        .join(google, how="left")

In [33]:
df.head()

risk_3   risk_12  logfat_risk_3  logfat_risk_12  INFORM  \
iso3 month                                                                   
AFG  2018-01-01  0.999608  1.000000       8.331437        9.280007     7.7   
     2018-02-01  0.996514  1.000000       8.295601        9.393767     7.7   
     2018-03-01  0.998218  1.000000       8.339758        9.346226     7.7   
     2018-04-01  0.995082  0.999815       8.383570        9.414723     7.7   
     2018-05-01  0.991237  0.998947       8.342653        9.481498     7.7   

                  VU   CC   HA  \
iso3 month                       
AFG  2018-01-01  7.1  7.5  8.7   
     2018-02-01  7.1  7.5  8.7   
     2018-03-01  7.1  7.5  8.7   
     2018-04-01  7.1  7.5  8.7   
     2018-05-01  7.1  7.5  8.7   

                                                        event_type  \
iso3 month                                                           
AFG  2018-01-01  [Battles, Explosions/Remote violence, Strategi...   
     2018-02-01  [Battles, Explosions/Remote violence, Strategi...   
     2018-03-01  [Battles, Explosions/Remote violence, Violence...   
     2018-04-01  [Battles, Explosions/Remote violence, Protests...   
     2018-05-01  [Battles, Explosions/Remote violence, Strategi...   

                                                    sub_event_type  ...  \
iso3 month                                                          ...   
AFG  2018-01-01  [Armed clash, Government regains territory, Ai...  ...   
     2018-02-01  [Armed clash, Air/drone strike, Grenade, Remot...  ...   
     2018-03-01  [Armed clash, Air/drone strike, Remote explosi...  ...   
     2018-04-01  [Armed clash, Non-state actor overtakes territ...  ...   
     2018-05-01  [Armed clash, Non-state actor overtakes territ...  ...   

                hdx_value  monthly_displacement  Flight Airport  Travel  \
iso3 month                                                                
AFG  2018-01-01       NaN           7456.583577     4.0     2.0     2.0   
     2018-02-01       1.0          13918.956010     3.0     2.0     2.0   
     2018-03-01       NaN          15410.272726     4.0     2.0     2.0   
     2018-04-01       NaN          17198.881440     3.0     2.0     2.0   
     2018-05-01       2.0          39769.719735     4.0     2.0     2.0   

                 Train  Bus  Passport  Travel visa  Right of asylum  
iso3 month                                                           
AFG  2018-01-01    1.0  1.0       1.0          4.0              0.0  
     2018-02-01    1.0  1.0       1.0          4.0              0.0  
     2018-03-01    0.0  1.0       1.0          4.0              0.0  
     2018-04-01    1.0  1.0       1.0          5.0              0.0  
     2018-05-01    1.0  1.0       1.0          5.0              0.0  

[5 rows x 24 columns]

In [34]:
df.size, df.shape

(418176, (17424, 24))

In [35]:
print(f"Min month: {df.index.get_level_values('month').min()}")
print(f"Max month: {df.index.get_level_values('month').max()}")
print(f"Number of countries: {df.index.get_level_values('iso3').nunique()}")

Min month: 2018-01-01
Max month: 2026-03-01
Number of countries: 176


Let's now solve the problem of the missing values:

- The missing values in `disorder_type`, `sub_event_type`, and `event_type` correspond to month x country that hasen't had any event. Then we will put a empty list.

- Missing values in `events` and `fatalities` correspond to months x countries with no events nor fatalities, so we will put 0 on both. 

- Missing values in `monthly_displacement` correspond to countries that hasn't had any displacement in all the covered period, so we are putting a 0.

- Missing values in `hdx_alert_level` correspond to the month x country with no hdx alert, so we put an empty list. Te same for `hdx_value`, so we are putting a 0.

In [36]:
df["event_type"] = df["event_type"].apply(
    lambda x: [] if x is None or (isinstance(x, float) and pd.isna(x)) else x
)
df["sub_event_type"] = df["sub_event_type"].apply(
    lambda x: [] if x is None or (isinstance(x, float) and pd.isna(x)) else x
)
df["disorder_type"] = df["disorder_type"].apply(
    lambda x: [] if x is None or (isinstance(x, float) and pd.isna(x)) else x
)
df["events"] = df["events"].fillna(0)
df["fatalities"] = df["fatalities"].fillna(0)
df["hdx_alert_level"] = df["hdx_alert_level"].apply(
    lambda x: [] if x is None or (isinstance(x, float) and pd.isna(x)) else x
)
df["hdx_value"] = df["hdx_value"].fillna(0)
df["monthly_displacement"] = df["monthly_displacement"].fillna(0)

In [37]:
df = df.sort_index()
df.head()

risk_3   risk_12  logfat_risk_3  logfat_risk_12  INFORM  \
iso3 month                                                                   
AFG  2018-01-01  0.999608  1.000000       8.331437        9.280007     7.7   
     2018-02-01  0.996514  1.000000       8.295601        9.393767     7.7   
     2018-03-01  0.998218  1.000000       8.339758        9.346226     7.7   
     2018-04-01  0.995082  0.999815       8.383570        9.414723     7.7   
     2018-05-01  0.991237  0.998947       8.342653        9.481498     7.7   

                  VU   CC   HA  \
iso3 month                       
AFG  2018-01-01  7.1  7.5  8.7   
     2018-02-01  7.1  7.5  8.7   
     2018-03-01  7.1  7.5  8.7   
     2018-04-01  7.1  7.5  8.7   
     2018-05-01  7.1  7.5  8.7   

                                                        event_type  \
iso3 month                                                           
AFG  2018-01-01  [Battles, Explosions/Remote violence, Strategi...   
     2018-02-01  [Battles, Explosions/Remote violence, Strategi...   
     2018-03-01  [Battles, Explosions/Remote violence, Violence...   
     2018-04-01  [Battles, Explosions/Remote violence, Protests...   
     2018-05-01  [Battles, Explosions/Remote violence, Strategi...   

                                                    sub_event_type  ...  \
iso3 month                                                          ...   
AFG  2018-01-01  [Armed clash, Government regains territory, Ai...  ...   
     2018-02-01  [Armed clash, Air/drone strike, Grenade, Remot...  ...   
     2018-03-01  [Armed clash, Air/drone strike, Remote explosi...  ...   
     2018-04-01  [Armed clash, Non-state actor overtakes territ...  ...   
     2018-05-01  [Armed clash, Non-state actor overtakes territ...  ...   

                hdx_value  monthly_displacement  Flight Airport  Travel  \
iso3 month                                                                
AFG  2018-01-01       0.0           7456.583577     4.0     2.0     2.0   
     2018-02-01       1.0          13918.956010     3.0     2.0     2.0   
     2018-03-01       0.0          15410.272726     4.0     2.0     2.0   
     2018-04-01       0.0          17198.881440     3.0     2.0     2.0   
     2018-05-01       2.0          39769.719735     4.0     2.0     2.0   

                 Train  Bus  Passport  Travel visa  Right of asylum  
iso3 month                                                           
AFG  2018-01-01    1.0  1.0       1.0          4.0              0.0  
     2018-02-01    1.0  1.0       1.0          4.0              0.0  
     2018-03-01    0.0  1.0       1.0          4.0              0.0  
     2018-04-01    1.0  1.0       1.0          5.0              0.0  
     2018-05-01    1.0  1.0       1.0          5.0              0.0  

[5 rows x 24 columns]

### CREATE THE TARGET VARIABLE

Our target variable is going to be an incidence variable showing if there's going to be a situation for which the country is elegible for an allocation in the next 2 months. To do so, we first need to create the variable allocation-elegible.

We say that a country is elegible to recieve an allocation if the country satisfies the necessary conditions for the CERF to send an allocation: 50,000 new internal displacements over a 3 months period.

Our target variable predicts whether in the following two months there's going to be a situation for whitch the CERF would send allocation. Eventhough we are interseted in predicting the first month for which this state is met, we don't want to put a 0 in the target variable, if in the next two months the condition is met but not for the first time, because that will confuse a lot our model. The other option was to put Nan in thoose cases, but considering that we have a super balanced datset, we choose to put 1.

In [38]:
df = df.reset_index()
df = df.sort_values(by=['iso3', 'month'])


df['rolling_3m_displacements'] = (
    df.groupby('iso3')['monthly_displacement']
    .rolling(window=3, min_periods=1)
    .sum()
    .reset_index(level=0, drop=True)
)

df['allocation-eligible'] = (df['rolling_3m_displacements'] >= 50000).astype(int)
df = df.set_index(['iso3', 'month']).sort_index()

Now we can create our target variable:

In [39]:
# Create the target variable: 1 if conflict in t+1 or t+2, else 0
df["target_2m"] = (
    (df.groupby(level="iso3")["allocation-eligible"].shift(-1) == 1) |
    (df.groupby(level="iso3")["allocation-eligible"].shift(-2) == 1)
).astype(int)

print("Total positive targets (conflict in next 2 months):", df["target_2m"].sum())

Total positive targets (conflict in next 2 months): 777


### FEATURE ENGINEERING

#### CERF VARIABLES

We know from the CERF that depending on the state of a country in a determinate moment, it's susceptible of having one type of conflict or another (hard onset or pretracted). The important thing is that depending on the state, the early signals that indicates an incoming crisis are different. So the first thing that we have to do is classify each county x month to one of the two states, and then, depending on it, check which signals are beeing seen. 

In order to classify a country x month in a state, we use the CERF logic, saying:

"A conflict is classified as a “protracted crisis” when EconAI’s risk score remains consistently above 0.6 over a period of 12 months."

Then, the early signals depending on the state are:

1. Protracted: 
- EconAI’s risk score increases by ≥0.1 over 3 consecutive months while already >0.7; OR
- Monthly displacement or fatalities increase by ≥1.5 times rolling 6-month average for 2 consecutive months; OR
- There are ≥3 Medium or High HDX signals occur within 90 days.

2. Hard onset:
- EconAI’s risk score increases by ≥0.3 within 2 months (this will typically also be confirmed by the issuance of an HDX signal the month of the significant increase); OR
- Monthly displacement increases by > 3 times the rolling 6-month average (this will typically be confirmed by a sharp increase in risk score). 

In instances where displacement data is missing, such as Armenia, the fatalities trend can be used instead.


In [13]:
# 1. CLASSIFY THE STATE (Protracted vs Hard Onset)

# Rule: Risk > 0.6 for 12 consecutive months
df['risk_gt_06'] = df['risk'] > 0.6

# We use rolling sum of 12 on the boolean column. If sum is 12, it was True for 12 consecutive months.
df['is_protracted'] = df.groupby(level='iso3')['risk_gt_06'].transform(
    lambda x: x.rolling(window=12, min_periods=12).sum() == 12
).astype(int)

df['state'] = np.where(df['is_protracted']==1, 'Protracted', 'Hard onset')

In [14]:
# 2. PRE-CALCULATE BASE METRICS

# 6-month rolling averages
df['disp_6m_avg'] = df.groupby(level='iso3')['monthly_displacement'].transform(lambda x: x.rolling(6).mean())
df['fat_6m_avg'] = df.groupby(level='iso3')['fatalities'].transform(lambda x: x.rolling(6).mean())

# Function to safely parse the HDX alert level list and count 'Medium' or 'High'
def count_hdx_med_high(x):
    if isinstance(x, list):
        return sum(1 for item in x if item in ['Medium concern', 'High concern'])
    elif isinstance(x, str): # In case pandas read the list as a string
        return 1 if 'Medium concern' in x or 'High concern' in x else 0
    return 0

df['hdx_med_high_count'] = df['hdx_alert_level'].apply(count_hdx_med_high)
# Rolling 90 days (3 months) sum of HDX alerts
df['hdx_3m_sum'] = df.groupby(level='iso3')['hdx_med_high_count'].transform(lambda x: x.rolling(3).sum())

In [15]:
# 3. PROTRACTED SIGNALS EVALUATION

# P1: Risk increases by >= 0.1 over 3 months while already > 0.7
df['p_sig1'] = (df.groupby(level='iso3')['risk'].diff(3) >= 0.1) & (df['risk'] > 0.7)

# P2: Displacement or fatalities >= 1.5x their 6-month average for 2 consecutive months
spike_1_5x = (df['monthly_displacement'] >= 1.5 * df['disp_6m_avg']) | (df['fatalities'] >= 1.5 * df['fat_6m_avg'])

# Agrupamos la Serie directamente usando su propio índice (level='iso3')
df['p_sig2'] = spike_1_5x.groupby(level='iso3').transform(
    lambda x: x.rolling(2).sum() == 2
)
# P3: >= 3 Medium or High HDX signals within 90 days
df['p_sig3'] = df['hdx_3m_sum'] >= 3

# Combine Protracted signals (True if any is True)
df['protracted_signal'] = df['p_sig1'] | df['p_sig2'] | df['p_sig3']


In [16]:
# 4. HARD ONSET SIGNALS EVALUATION

# H1: Risk increases by >= 0.3 within 2 months
df['h_sig1'] = df.groupby(level='iso3')['risk'].diff(2) >= 0.3

# H2: Displacement > 3x 6-month avg. If missing, use fatalities > 3x 6-month avg.
disp_spike_3x = df['monthly_displacement'] > 3 * df['disp_6m_avg']
fat_spike_3x = df['fatalities'] > 3 * df['fat_6m_avg']

# np.where lets us use the fatality logic ONLY when displacement is NaN
df['h_sig2'] = np.where(df['monthly_displacement'].isna(), fat_spike_3x, disp_spike_3x)

# Combine Hard Onset signals
df['hard_onset_signal'] = df['h_sig1'] | df['h_sig2']


In [17]:
# 5. FINAL EARLY SIGNAL COLUMN

# Apply the corresponding signal based on the state of the country that month
df['early_signal'] = np.where(
    df['state'] == 'Protracted',
    df['protracted_signal'],
    df['hard_onset_signal']
).astype(int) # Convert True/False to 1/0

# Let's check the distribution!
print(df['state'].value_counts())
print(f"\nTotal early signals detected: {df['early_signal'].sum()}")

df = df.drop(columns=['state'])

state
Hard onset    7192
Protracted    1652
Name: count, dtype: int64

Total early signals detected: 827


In [18]:
df.head()

monthly_displacement  fatalities  population_5km  events  \
iso3 month                                                                  
AB9  2018-01-01                   0.0         0.0             0.0     0.0   
     2018-02-01                   0.0         0.0             0.0     0.0   
     2018-03-01                   0.0         0.0             0.0     0.0   
     2018-04-01                   0.0         0.0             0.0     0.0   
     2018-05-01                   0.0         0.0             0.0     0.0   

                disorder_type event_type sub_event_type interaction notes  \
iso3 month                                                                  
AB9  2018-01-01            []         []             []          []    []   
     2018-02-01            []         []             []          []    []   
     2018-03-01            []         []             []          []    []   
     2018-04-01            []         []             []          []    []   
     2018-05-01            []         []             []          []    []   

                hdx_alert_level  ...  hdx_med_high_count  hdx_3m_sum  p_sig1  \
iso3 month                       ...                                           
AB9  2018-01-01              []  ...                   0         NaN   False   
     2018-02-01              []  ...                   0         NaN   False   
     2018-03-01              []  ...                   0         0.0   False   
     2018-04-01              []  ...                   0         0.0   False   
     2018-05-01              []  ...                   0         0.0   False   

                 p_sig2  p_sig3  protracted_signal  h_sig1  h_sig2  \
iso3 month                                                           
AB9  2018-01-01   False   False              False   False   False   
     2018-02-01   False   False              False   False   False   
     2018-03-01   False   False              False   False   False   
     2018-04-01   False   False              False   False   False   
     2018-05-01   False   False              False   False   False   

                 hard_onset_signal  early_signal  
iso3 month                                        
AB9  2018-01-01              False             0  
     2018-02-01              False             0  
     2018-03-01              False             0  
     2018-04-01              False             0  
     2018-05-01              False             0  

[5 rows x 43 columns]

In [19]:
display(pd.DataFrame(df.columns, columns=['Column Name']))

,Column Name
0,monthly_displacement
1,fatalities
2,population_5km
3,events
4,disorder_type
5,event_type
6,sub_event_type
7,interaction
8,notes
9,hdx_alert_level


In [20]:
# Llista de columnes a les que volem aplicar el lag
cols_to_lag = ['monthly_displacement', 'fatalities', 'events', 'risk']

for col in cols_to_lag:
    # Creem el Lag 1 (mes anterior)
    df[f'{col}_lag1'] = df.groupby('iso3')[col].shift(1)
    
    # Creem el Lag 2 (fa dos mesos) - Opcional però recomanat per a displacement
    if col == 'monthly_displacement':
        df[f'{col}_lag2'] = df.groupby('iso3')[col].shift(2)

#### NEW VARIABLES

Now we have to use the variables that are categoricat in a way that are usefull. Not using one-hot encoding in all of them.

In [21]:
import ast

def safe_parse_list(val):
    # 1. Si ja és una llista (perquè hem processat la dada abans), no facis res
    if isinstance(val, list):
        return val
        
    # 2. Si és un string, mirem si té format de llista: "[item1, item2]"
    if isinstance(val, str):
        val = val.strip()
        if val.startswith('[') and val.endswith(']'):
            try:
                # ast.literal_eval converteix el text "[1, 2]" en la llista real [1, 2]
                parsed = ast.literal_eval(val)
                return parsed if isinstance(parsed, list) else [parsed]
            except (ValueError, SyntaxError):
                return []
        else:
            # Si és un string normal ("Attack"), el posem dins d'una llista ["Attack"]
            return [val] if val != "" else []
            
    # 3. Si és un NaN o un valor buit, retorna una llista buida per no donar error
    return []

In [22]:

# 1. ORDINAL ENCODING per a HDX (Més intel·ligent que Dummies)
# Assignem un valor numèric perquè el model entengui la jerarquia de gravetat
hdx_mapping = {
    'None': 0,
    'Low concern': 1,
    'Medium concern': 2,
    'High concern': 3
}

def get_max_hdx_score(val_list):
    scores = [hdx_mapping.get(str(x), 0) for x in val_list]
    return max(scores) if scores else 0

df['hdx_alert_score'] = df['hdx_alert_level'].apply(safe_parse_list).apply(get_max_hdx_score)

# 2. AGRUPACIÓ AMB NOMS EXACTES (Basat en la teva llista)
def extract_event_features(row):
    # safe_parse_list ens torna la llista de noms exactes
    events = safe_parse_list(row['event_type'])
    sub_events = safe_parse_list(row['sub_event_type'])
    all_ev = set(events + sub_events)
    
    return pd.Series({
        # Violència contra civils (Noms exactes de la teva llista)
        'is_civilian_targeted': 1 if any(x in all_ev for x in [
            'Violence against civilians', 'Attack', 'Sexual violence', 
            'Abduction/forced disappearance', 'Looting/property destruction', 
            'Mob violence', 'Excessive force against protesters'
        ]) else 0,
        
        # Guerra i explosions (Noms exactes de la teva llista)
        'is_heavy_warfare': 1 if any(x in all_ev for x in [
            'Battles', 'Explosions/Remote violence', 'Armed clash', 
            'Air/drone strike', 'Shelling/artillery/missile attack', 
            'Grenade', 'Chemical weapon', 'Suicide bomb', 
            'Remote explosive/landmine/IED'
        ]) else 0,
        
        # Control territorial (Noms exactes de la teva llista)
        'is_territorial_shift': 1 if any(x in all_ev for x in [
            'Government regains territory', 'Non-state actor overtakes territory', 
            'Non-violent transfer of territory', 'Headquarters or base established'
        ]) else 0,
        
        # Protestes i inestabilitat (Noms exactes de la teva llista)
        'is_social_unrest': 1 if any(x in all_ev for x in [
            'Protests', 'Riots', 'Peaceful protest', 'Violent demonstration', 
            'Protest with intervention', 'Arrests'
        ]) else 0
    })

# 3. INTERACTION (Noms exactes de la teva llista de 'interaction')
def extract_interaction_features(val_list):
    interactions = safe_parse_list(val_list)
    # Aquí busquem els strings exactes que m'has passat
    full_str = " ".join([str(x) for x in interactions])
    
    return pd.Series({
        # Casos clau on apareix l'estat, rebels o civils
        'state_involved': 1 if 'State forces' in full_str else 0,
        'state_vs_civilians': 1 if ('State forces' in full_str and 'Civilians' in full_str) else 0,
        'rebel_involved': 1 if 'Rebel group' in full_str else 0,
        'militia_involved': 1 if 'militia' in full_str.lower() else 0,
        'external_forces_involved': 1 if 'External/Other forces' in full_str else 0,
        'protesters_only_interaction': 1 if 'Protesters only' in full_str else 0
    })

# Apliquem de nou amb el rigor dels noms que m'has donat
df_events = df.apply(extract_event_features, axis=1)
df_inter = df['interaction'].apply(extract_interaction_features)

df = pd.concat([df, df_events, df_inter], axis=1)


In [23]:
# Muestra las columnas en una tabla vertical interactiva
display(pd.DataFrame(df.columns, columns=['Column Name']))

,Column Name
0,monthly_displacement
1,fatalities
2,population_5km
3,events
4,disorder_type
5,event_type
6,sub_event_type
7,interaction
8,notes
9,hdx_alert_level


In [24]:
df.columns = df.columns.str.replace(" ", "_")

In [25]:
# Muestra las columnas en una tabla vertical interactiva
display(pd.DataFrame(df.columns, columns=['Column Name']))

,Column Name
0,monthly_displacement
1,fatalities
2,population_5km
3,events
4,disorder_type
5,event_type
6,sub_event_type
7,interaction
8,notes
9,hdx_alert_level


#### ADDING INFORMATION FROM THE NOTES

Let's now construct the index of violence scalation from the `notes` variable. The methodology is based on doing embeddings of the notes, and use the vector representation of those as features. We are not constructing and scalation of violence index, based for example in zero-shot, because the semantics depend a lot on the countries, so it would probably be unstable.

We are using the pretrained model BERT.

In [26]:
df[df["notes"].apply(lambda x: x != [])]["notes"].head(50)

iso3  month     
AFG   2018-01-01    [Infighting occurred between 2 groups of the T...
      2018-02-01    [As reported on February 25th, at least 4 unid...
      2018-03-01    [Between March 30th and March 31st, 11 of the ...
      2018-04-01    [As reported on April 27th, 7 Taliban militant...
      2018-05-01    [People staged a protest in front of the UN of...
      2018-06-01    [A number of tribal elders, civil society acti...
      2018-07-01    [The Taliban abducted a money-changer and held...
      2018-08-01    [Detonation: On 11-August-2018 3 Taliban insur...
      2018-09-01    [Unclaimed bombings occurred in front of 8 sch...
      2018-10-01    [Detonation: On 25-October-2018, 1 Taliban mil...
      2018-11-01    [Detonation: On 30-November-2018, 37 Taliban m...
      2018-12-01    [On 31 December 2018, 1 civilian was killed fo...
      2019-01-01    [Property destruction: On 30-January-2019, 1 g...
      2019-02-01    [Detonation: On 18-February-2019, 2 unidentifi...
   

In [27]:
if EMB:    
    from sentence_transformers import SentenceTransformer

    model = SentenceTransformer("all-MiniLM-L6-v2")  # 384 dims

In [28]:
if EMB:
    from tqdm import tqdm
    df_exploded = df.explode("notes")
    unique_texts = (
        df_exploded["notes"]
        .dropna()
        .astype(str)
        .tolist()
    )

    embeddings = model.encode(
        unique_texts,
        batch_size=64,       
        show_progress_bar=True,
        convert_to_numpy=True
    )

    text_map = dict(zip(unique_texts, embeddings))
    df_exploded["embedding"] = df_exploded["notes"].map(text_map)

In [29]:
if EMB:   
    def aggregate_logic(series):
        valid_embs = [e for e in series if e is not None and not isinstance(e, float)]
        
        if not valid_embs:
            # Fill of zeros if no text in the whole month/country (384 * 2 = 768 dims)
            return np.zeros(768)
        
        matrix = np.vstack(valid_embs)
        
        # Compute the two "textual climate" metrics
        mean_pool = np.mean(matrix, axis=0)
        norms = np.linalg.norm(matrix, axis=1)
        max_pool = matrix[np.argmax(norms)]
        
        return np.concatenate([mean_pool, max_pool])

    df_clima = df_exploded.groupby(level=["iso3", "month"])["embedding"].apply(aggregate_logic)
    df_clima.to_csv("../data_clean/emb.csv")

In [30]:
df_clima = pd.read_csv("../data_clean/emb.csv")
df_clima = (
    df_clima
    .rename(columns={"embedding": "notes_vector"})
    .set_index(['iso3', 'month'])
    .sort_index()
)
df_final = df.join(df_clima)

In [31]:
df_final.head()

monthly_displacement  fatalities  population_5km  events  \
iso3 month                                                                  
AB9  2018-01-01                   0.0         0.0             0.0     0.0   
     2018-02-01                   0.0         0.0             0.0     0.0   
     2018-03-01                   0.0         0.0             0.0     0.0   
     2018-04-01                   0.0         0.0             0.0     0.0   
     2018-05-01                   0.0         0.0             0.0     0.0   

                disorder_type event_type sub_event_type interaction notes  \
iso3 month                                                                  
AB9  2018-01-01            []         []             []          []    []   
     2018-02-01            []         []             []          []    []   
     2018-03-01            []         []             []          []    []   
     2018-04-01            []         []             []          []    []   
     2018-05-01            []         []             []          []    []   

                hdx_alert_level  ...  is_heavy_warfare  is_territorial_shift  \
iso3 month                       ...                                           
AB9  2018-01-01              []  ...                 0                     0   
     2018-02-01              []  ...                 0                     0   
     2018-03-01              []  ...                 0                     0   
     2018-04-01              []  ...                 0                     0   
     2018-05-01              []  ...                 0                     0   

                 is_social_unrest  state_involved  state_vs_civilians  \
iso3 month                                                              
AB9  2018-01-01                 0               0                   0   
     2018-02-01                 0               0                   0   
     2018-03-01                 0               0                   0   
     2018-04-01                 0               0                   0   
     2018-05-01                 0               0                   0   

                 rebel_involved  militia_involved  external_forces_involved  \
iso3 month                                                                    
AB9  2018-01-01               0                 0                         0   
     2018-02-01               0                 0                         0   
     2018-03-01               0                 0                         0   
     2018-04-01               0                 0                         0   
     2018-05-01               0                 0                         0   

                 protesters_only_interaction  \
iso3 month                                     
AB9  2018-01-01                            0   
     2018-02-01                            0   
     2018-03-01                            0   
     2018-04-01                            0   
     2018-05-01                            0   

                                                      notes_vector  
iso3 month                                                          
AB9  2018-01-01  [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. ...  
     2018-02-01  [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. ...  
     2018-03-01  [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. ...  
     2018-04-01  [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. ...  
     2018-05-01  [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. ...  

[5 rows x 60 columns]

In [32]:
df_final = df_final.sort_index(level=['iso3', 'month'])
display(pd.DataFrame(df_final.columns, columns=['Column Name']))

,Column Name
0,monthly_displacement
1,fatalities
2,population_5km
3,events
4,disorder_type
5,event_type
6,sub_event_type
7,interaction
8,notes
9,hdx_alert_level


In [33]:
# Convert the 'notes_vector' column from string to numpy array
def clean_and_parse_vector(x):
    # Si és una llista o un array de veritat i té elements
    if isinstance(x, (list, np.ndarray)):
        if len(x) == 768:
            return np.array(x)
        elif len(x) == 0:
            return np.zeros(768)  # Si està buit [], omplim amb 768 zeros
            
    # Si el vector ha quedat guardat com a text (String) per error
    if isinstance(x, str):
        x = x.strip().replace('[', '').replace(']', '')
        if not x:  # Si és un text buit o només claudàtors "[]"
            return np.zeros(768)
        # Convertim el text separat per espais o comes en floats
        vals = [float(num) for num in x.split() if num.strip()]
        if len(vals) == 768:
            return np.array(vals)
            
    # Per a qualsevol altre cas estrany (NaNs, Nulls, etc.)
    return np.zeros(768)

# Apliquem la neteja a tota la columna
df_final['notes_vector'] = df_final['notes_vector'].apply(clean_and_parse_vector)

In [34]:
# Convertim la columna d'arrays en un DataFrame de 768 columnes
df_vectors = pd.DataFrame(df_final['notes_vector'].tolist(), index=df_final.index)

# Posem noms clars: 0-383 són Mean, 384-767 són Max
df_vectors.columns = [f'emb_mean_{i}' for i in range(384)] + [f'emb_max_{i}' for i in range(384)]

# L'unim al dataframe principal i eliminem la columna original 'notes_vector'
df_final = pd.concat([df_final, df_vectors], axis=1)

In [35]:
from sklearn.decomposition import PCA

# Identifiquem les columnes de cada tipus
cols_mean = [c for c in df_final.columns if c.startswith('emb_mean_')]
cols_max = [c for c in df_final.columns if c.startswith('emb_max_')]

# PCA per a la Mitjana (per exemple, 30 components)
pca_mean = PCA(n_components=5)
res_mean = pca_mean.fit_transform(df_final[cols_mean])

# PCA per al Màxim (altres 30 components)
pca_max = PCA(n_components=5)
res_max = pca_max.fit_transform(df_final[cols_max])

In [36]:
# Creem els noms de les columnes per saber què és cada cosa
cols_pca_mean = [f'pca_mean_{i}' for i in range(5)]
cols_pca_max = [f'pca_max_{i}' for i in range(5)]

# Construïm el DataFrame de resultats
df_pca = pd.DataFrame(
    np.hstack([res_mean, res_max]), 
    index=df_final.index, 
    columns=cols_pca_mean + cols_pca_max
)

In [37]:
df_pca.head()

pca_mean_0  pca_mean_1  pca_mean_2  pca_mean_3  pca_mean_4  \
iso3 month                                                                    
AB9  2018-01-01    0.423957   -0.013541    0.003866   -0.000654   -0.009994   
     2018-02-01    0.423957   -0.013541    0.003866   -0.000654   -0.009994   
     2018-03-01    0.423957   -0.013541    0.003866   -0.000654   -0.009994   
     2018-04-01    0.423957   -0.013541    0.003866   -0.000654   -0.009994   
     2018-05-01    0.423957   -0.013541    0.003866   -0.000654   -0.009994   

                 pca_max_0  pca_max_1  pca_max_2  pca_max_3  pca_max_4  
iso3 month                                                              
AB9  2018-01-01   0.168423   0.385689   0.009678   0.026924   0.005205  
     2018-02-01   0.168423   0.385689   0.009678   0.026924   0.005205  
     2018-03-01   0.168423   0.385689   0.009678   0.026924   0.005205  
     2018-04-01   0.168423   0.385689   0.009678   0.026924   0.005205  
     2018-05-01   0.168423   0.385689   0.009678   0.026924   0.005205

In [38]:
df_final = pd.concat([df_final, df_pca], axis=1)

In [39]:
display(pd.DataFrame(df_final.columns, columns=['Column Name']))

,Column Name
0,monthly_displacement
1,fatalities
2,population_5km
3,events
4,disorder_type
...,...
833,pca_max_0
834,pca_max_1
835,pca_max_2
836,pca_max_3


#### FEATURES FROM GOOGLE TRENDS

In [40]:
# Primer, assegura't que les dades estan ordenades cronològicament per país
features_to_lag = ['Flight', 'Airport', 'Travel', 'Train', 'Bus', 'Passport', 'Travel_visa', 'Right_of_asylum']

# Creem lags d'1, 2 i 3 mesos enrere de manera agrupada per país (per no barrejar dades)
for lag in [1, 2, 3]:
    for col in features_to_lag:
        df_final[f'{col}_lag_{lag}m'] = df_final.groupby('iso3')[col].shift(lag)


# Ràtio de creixement respecte al mes anterior (Acceleració)
for col in features_to_lag:
    # Afegim un petit epsilon (+ 1e-5) per evitar la divisió per zero si la cerca era 0
    df_final[f'{col}_growth_1m'] = (df_final[col] - df_final.groupby('iso3')[col].shift(1)) / (df_final.groupby('iso3')[col].shift(1) + 1e-5)


# Mitjana mòbil dels últims 3 mesos
for col in features_to_lag:
    df_final[f'{col}_rolling_mean_3m'] = df_final.groupby('iso3')[col].transform(lambda x: x.rolling(window=3, min_periods=1).mean())

In [41]:
df_final.head()

monthly_displacement  fatalities  population_5km  events  \
iso3 month                                                                  
AB9  2018-01-01                   0.0         0.0             0.0     0.0   
     2018-02-01                   0.0         0.0             0.0     0.0   
     2018-03-01                   0.0         0.0             0.0     0.0   
     2018-04-01                   0.0         0.0             0.0     0.0   
     2018-05-01                   0.0         0.0             0.0     0.0   

                disorder_type event_type sub_event_type interaction notes  \
iso3 month                                                                  
AB9  2018-01-01            []         []             []          []    []   
     2018-02-01            []         []             []          []    []   
     2018-03-01            []         []             []          []    []   
     2018-04-01            []         []             []          []    []   
     2018-05-01            []         []             []          []    []   

                hdx_alert_level  ...  Travel_visa_growth_1m  \
iso3 month                       ...                          
AB9  2018-01-01              []  ...                    NaN   
     2018-02-01              []  ...                    NaN   
     2018-03-01              []  ...                    NaN   
     2018-04-01              []  ...                    NaN   
     2018-05-01              []  ...                    NaN   

                 Right_of_asylum_growth_1m  Flight_rolling_mean_3m  \
iso3 month                                                           
AB9  2018-01-01                        NaN                     NaN   
     2018-02-01                        NaN                     NaN   
     2018-03-01                        NaN                     NaN   
     2018-04-01                        NaN                     NaN   
     2018-05-01                        NaN                     NaN   

                 Airport_rolling_mean_3m  Travel_rolling_mean_3m  \
iso3 month                                                         
AB9  2018-01-01                      NaN                     NaN   
     2018-02-01                      NaN                     NaN   
     2018-03-01                      NaN                     NaN   
     2018-04-01                      NaN                     NaN   
     2018-05-01                      NaN                     NaN   

                 Train_rolling_mean_3m  Bus_rolling_mean_3m  \
iso3 month                                                    
AB9  2018-01-01                    NaN                  NaN   
     2018-02-01                    NaN                  NaN   
     2018-03-01                    NaN                  NaN   
     2018-04-01                    NaN                  NaN   
     2018-05-01                    NaN                  NaN   

                 Passport_rolling_mean_3m  Travel_visa_rolling_mean_3m  \
iso3 month                                                               
AB9  2018-01-01                       NaN                          NaN   
     2018-02-01                       NaN                          NaN   
     2018-03-01                       NaN                          NaN   
     2018-04-01                       NaN                          NaN   
     2018-05-01                       NaN                          NaN   

                 Right_of_asylum_rolling_mean_3m  
iso3 month                                        
AB9  2018-01-01                              NaN  
     2018-02-01                              NaN  
     2018-03-01                              NaN  
     2018-04-01                              NaN  
     2018-05-01                              NaN  

[5 rows x 878 columns]

In [42]:
print(df_final.columns.tolist())

['monthly_displacement', 'fatalities', 'population_5km', 'events', 'disorder_type', 'event_type', 'sub_event_type', 'interaction', 'notes', 'hdx_alert_level', 'hdx_value', 'risk', 'logfat_risk', 'any_violence_risk', 'Flight', 'Airport', 'Travel', 'Train', 'Bus', 'Passport', 'Travel_visa', 'Right_of_asylum', 'INFORM', 'VU', 'CC', 'HA', 'rolling_3m_displacements', 'conflict', 'conflict_2m', 'risk_gt_06', 'is_protracted', 'disp_6m_avg', 'fat_6m_avg', 'hdx_med_high_count', 'hdx_3m_sum', 'p_sig1', 'p_sig2', 'p_sig3', 'protracted_signal', 'h_sig1', 'h_sig2', 'hard_onset_signal', 'early_signal', 'monthly_displacement_lag1', 'monthly_displacement_lag2', 'fatalities_lag1', 'events_lag1', 'risk_lag1', 'hdx_alert_score', 'is_civilian_targeted', 'is_heavy_warfare', 'is_territorial_shift', 'is_social_unrest', 'state_involved', 'state_vs_civilians', 'rebel_involved', 'militia_involved', 'external_forces_involved', 'protesters_only_interaction', 'notes_vector', 'emb_mean_0', 'emb_mean_1', 'emb_mean_2

In [43]:
df_final.to_csv("../data_clean/final_text_data.csv")

In [45]:
import pandas as pd

# Suposem que el teu índex o les teves columnes tenen 'iso3' i 'month'
# Per si de cas estan com a índex (MultiIndex), els passem temporalment a columnes per analitzar-los millor
df_check = df_final.reset_index() if isinstance(df_final.index, pd.MultiIndex) else df_final.copy()

# Assegurem que la columna 'month' sigui de tipus datetime
df_check['month'] = pd.to_datetime(df_check['month'])

# 1. Comptar ISO3 i Months diferents
num_countries = df_check['iso3'].nunique()
num_months = df_check['month'].nunique()

print("=" * 50)
print(f"📊 ESTADÍSTIQUES DEL DATASET:")
print(f"   -> Nombre de països (iso3) únics: {num_countries}")
print(f"   -> Nombre de mesos únics totals:  {num_months}")
print("=" * 50)

# 2. Comprovar si hi ha Gaps (buits) en la línia temporal global
min_date = df_check['month'].min()
max_date = df_check['month'].max()

# Generem el rang teòric perfecte mes a mes sense cap salt
expected_range = pd.date_range(start=min_date, end=max_date, freq='MS') # 'MS' = Month Start

# Busquem si hi ha algun mes del rang teòric que NO existeixi a les nostres dades
missing_months = [m.strftime('%Y-%m-%d') for m in expected_range if m not in df_check['month'].values]

print(f"📅 ANÀLISI TEMPORAL (Des de {min_date.strftime('%Y-%m')} fins a {max_date.strftime('%Y-%m')}):")
print(f"   -> Mesos teòrics esperats: {len(expected_range)}")

if len(missing_months) == 0:
    print("   -> ✅ PERFECTE: No hi ha cap gap temporal al dataset global. Tots els mesos estan coberts.")
else:
    print(f"   -> ⚠️ ALERTA: S'han trobat {len(missing_months)} mesos absents en el global!")
    print(f"   -> Mesos que falten: {missing_months}")
print("=" * 50)

📊 ESTADÍSTIQUES DEL DATASET:
   -> Nombre de països (iso3) únics: 89
   -> Nombre de mesos únics totals:  100
📅 ANÀLISI TEMPORAL (Des de 2018-01 fins a 2026-04):
   -> Mesos teòrics esperats: 100
   -> ✅ PERFECTE: No hi ha cap gap temporal al dataset global. Tots els mesos estan coberts.


### PLOT THE RESULTS:

In [47]:
df_final = pd.read_csv("../data_clean/final_text_data.csv")

In [54]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go

def plot_full_crisis_timeline(df, cerf_path, country_code):
    # 1. PREPARAR EL DATAFRAME PRINCIPAL
    if 'iso3' not in df.columns:
        df = df.reset_index()
        
    df_plot = df[df['iso3'] == country_code].copy()
    df_plot['month'] = pd.to_datetime(df_plot['month'])
    df_plot = df_plot.sort_values('month')
    
    if df_plot.empty:
        print(f"No hay datos en el DataFrame principal para el país: {country_code}")
        return

    # Transformación Log(1 + x) para desplazamientos
    df_plot['disp_log1p'] = np.log1p(df_plot['monthly_displacement'].fillna(0))

    # Extraer las fechas de alerta conflict_2m (marcadas a mitad de mes)
    if 'conflict_2m' in df_plot.columns:
        warning_months = df_plot[df_plot['conflict_2m'] == 1]['month']
        warning_dates = warning_months + pd.Timedelta(days=0)
    else:
        warning_dates = pd.Series(dtype='datetime64[ns]')

    # 3. CREAR LA FIGURA INTERACTIVA
    fig = go.Figure()

    # --- B. Desplazamientos (Línea simple morada - Puesta en y1 para que NO desaparezca) ---
    fig.add_trace(go.Scatter(
        x=df_plot['month'],
        y=df_plot['disp_log1p'],
        customdata=df_plot['monthly_displacement'], 
        mode='lines', 
        name='Displacements [log(1+x)]',
        line=dict(color='purple', width=2.5),
        yaxis='y1',
        hovertemplate='<b>Date:</b> %{x|%b %Y}<br><b>Displacements (Real):</b> %{customdata:,.0f}<br><b>log(1+x):</b> %{y:.2f}<extra></extra>'
    ))

    # --- D. Zonas de Conflicto (Fondo sombreado salmón) ---
    if 'conflict' in df_plot.columns:
        conflict_months = df_plot[df_plot['conflict'] == 1]['month']
        for c_month in conflict_months:
            fig.add_vrect(
                x0=c_month - pd.Timedelta(days=15), 
                x1=c_month + pd.Timedelta(days=15),
                fillcolor="salmon",
                opacity=0.2,
                layer="below",
                line_width=0,
            )
        # Leyenda del conflicto
        fig.add_trace(go.Scatter(
            x=[None], y=[None], mode='lines', name='Conflict Zone',
            line=dict(color='salmon', width=10), opacity=0.3, yaxis='y1'
        ))

    # --- NUEVO: Condición >50k Desplazamientos (Fondo azul clarito) ---
    # Comprueba que tu columna se llame 'target'. Si se llama 'target_4m', cámbialo aquí.
    target_col = 'target' if 'target' in df_plot.columns else None
    if target_col:
        target_months = df_plot[df_plot[target_col] == 1]['month']
        for t_month in target_months:
            fig.add_vrect(
                x0=t_month - pd.Timedelta(days=15), 
                x1=t_month + pd.Timedelta(days=15),
                fillcolor="lightblue",
                opacity=0.3,
                layer="below",
                line_width=0,
            )
        fig.add_trace(go.Scatter(
            x=[None], y=[None], mode='lines', name='>50k Displacements (Target)',
            line=dict(color='lightblue', width=10), opacity=0.4, yaxis='y1'
        ))

    # --- F. Líneas conflict_2m (Líneas verticales rojas a rayas) ---
    if not warning_dates.empty:
        for w_date in warning_dates:
            fig.add_shape(
                type="line",
                x0=w_date, x1=w_date,
                y0=0, y1=1,
                xref="x", yref="paper",
                line=dict(width=2.5, dash="dash", color="red"),
            )
        # Leyenda de Alertas
        fig.add_trace(go.Scatter(
            x=[None], y=[None], mode='lines', name='Warning (1-2m Pre-Conflict)',
            line=dict(color='red', width=2.5, dash='dash'), yaxis='y1'
        ))

    # --- 4. CONFIGURACIÓN DEL LAYOUT (Eliminados los ejes invisibles) ---
    fig.update_layout(
        title=dict(
            text=f'<b>Full Crisis Timeline Overview: {country_code}</b>',
            font=dict(size=22),
            x=0.05
        ),
        margin=dict(l=60, r=100, t=110, b=60), 
        height=600,
        plot_bgcolor='white',
        hovermode="closest",
        
        xaxis=dict(
            title=dict(text="<b>Date</b>"),
            showgrid=False,
            dtick="M3",
            tickformat="%b %Y",
            tickangle=45
        ),
        
        # Eje Y1: Ahora son los Desplazamientos
        yaxis=dict(
            title=dict(text="<b>log(1 + Displacements)</b>", font=dict(color="purple")),
            tickfont=dict(color="purple"),
            showgrid=True,
            gridcolor='lightgrey',
            rangemode="tozero"
        ),
        
        legend=dict(
            orientation="h",
            yanchor="bottom", y=1.02,
            xanchor="center", x=0.5,
            bgcolor='rgba(255,255,255,0.8)',
            bordercolor='lightgrey',
            borderwidth=1
        )
    )

    fig.show()

# --- EJECUCIÓN ---
plot_full_crisis_timeline(df_final, cerf_path="../data_clean/cerf_clean.csv", country_code="SYR")

In [44]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go

def plot_full_crisis_timeline(df, cerf_path, country_code):
    # 1. PREPARAR EL DATAFRAME PRINCIPAL
    if 'iso3' not in df.columns:
        df = df.reset_index()
        
    df_plot = df[df['iso3'] == country_code].copy()
    df_plot['month'] = pd.to_datetime(df_plot['month'])
    df_plot = df_plot.sort_values('month')
    
    if df_plot.empty:
        print(f"No hay datos en el DataFrame principal para el país: {country_code}")
        return

    # Transformación Log(1 + x) para desplazamientos
    df_plot['disp_log1p'] = np.log1p(df_plot['monthly_displacement'].fillna(0))

    # 2. PREPARAR EL DATAFRAME DE CERF
    try:
        cerf = pd.read_csv(cerf_path)
        cerf = cerf[cerf['iso3'] == country_code].copy()
        cerf['Allocation Date'] = pd.to_datetime(cerf['Allocation Date'], format='ISO8601', errors='coerce')
        cerf = cerf.dropna(subset=['Allocation Date'])
    except FileNotFoundError:
        print(f"No se encontró el archivo CERF en {cerf_path}.")
        cerf = pd.DataFrame()

    # Extraer las fechas de alerta conflict_2m (marcadas a mitad de mes)
    if 'conflict_2m' in df_plot.columns:
        warning_months = df_plot[df_plot['conflict_2m'] == 1]['month']
        warning_dates = warning_months + pd.Timedelta(days=0)
    else:
        warning_dates = pd.Series(dtype='datetime64[ns]')

    # 3. CREAR LA FIGURA INTERACTIVA
    fig = go.Figure()

    # --- A. Fatalidades (Línea simple naranja - y1) ---
    fig.add_trace(go.Scatter(
        x=df_plot['month'],
        y=df_plot['fatalities'],
        mode='lines', 
        name='Fatalities',
        line=dict(color='orange', width=2.5),
        yaxis='y1',
        hovertemplate='<b>Date:</b> %{x|%b %Y}<br><b>Fatalities:</b> %{y:,.0f}<extra></extra>'
    ))

    # --- B. Desplazamientos (Línea simple morada - y2) ---
    fig.add_trace(go.Scatter(
        x=df_plot['month'],
        y=df_plot['disp_log1p'],
        customdata=df_plot['monthly_displacement'], 
        mode='lines', 
        name='Displacements [log(1+x)]',
        line=dict(color='purple', width=2.5),
        yaxis='y2',
        hovertemplate='<b>Date:</b> %{x|%b %Y}<br><b>Displacements (Real):</b> %{customdata:,.0f}<br><b>log(1+x):</b> %{y:.2f}<extra></extra>'
    ))

    # --- C. Riesgo (Línea Negra Punteada - y3) ---
    if 'risk' in df_plot.columns:
        fig.add_trace(go.Scatter(
            x=df_plot['month'],
            y=df_plot['risk'],
            mode='lines',
            name='Risk Score',
            line=dict(color='black', width=2, dash='dash'),
            yaxis='y3',
            hovertemplate='<b>Date:</b> %{x|%b %Y}<br><b>Risk:</b> %{y:.2f}<extra></extra>'
        ))

    # --- D. Zonas de Conflicto (Fondo sombreado salmón) ---
    if 'conflict' in df_plot.columns:
        conflict_months = df_plot[df_plot['conflict'] == 1]['month']
        for c_month in conflict_months:
            fig.add_vrect(
                x0=c_month - pd.Timedelta(days=15), 
                x1=c_month + pd.Timedelta(days=15),
                fillcolor="salmon",
                opacity=0.2,
                layer="below",
                line_width=0,
            )
        # Leyenda del conflicto
        fig.add_trace(go.Scatter(
            x=[None], y=[None], mode='lines', name='Conflict Zone',
            line=dict(color='salmon', width=10), opacity=0.3, yaxis='y1'
        ))

    # --- E. Líneas CERF (Líneas verticales verdes punteadas) ---
    if not cerf.empty:
        for _, row in cerf.iterrows():
            fig.add_shape(
                type="line",
                x0=row['Allocation Date'], x1=row['Allocation Date'],
                y0=0, y1=1,
                xref="x", yref="paper",
                line=dict(width=2.5, dash="dot", color="mediumseagreen"),
            )
        # Leyenda de CERF
        fig.add_trace(go.Scatter(
            x=[None], y=[None], mode='lines', name='CERF Allocation',
            line=dict(color='mediumseagreen', width=2.5, dash='dot'), yaxis='y1'
        ))

    # --- F. Líneas conflict_2m (Líneas verticales rojas a rayas) ---
    if not warning_dates.empty:
        for w_date in warning_dates:
            fig.add_shape(
                type="line",
                x0=w_date, x1=w_date,
                y0=0, y1=1,
                xref="x", yref="paper",
                line=dict(width=2.5, dash="dash", color="red"),
            )
        # Leyenda de Alertas
        fig.add_trace(go.Scatter(
            x=[None], y=[None], mode='lines', name='Warning (1-2m Pre-Conflict)',
            line=dict(color='red', width=2.5, dash='dash'), yaxis='y1'
        ))

   # --- G. HDX Alert Levels ---

    # --- G. HDX Alert Levels ---

    if 'hdx_alert_level' in df_plot.columns:

        hdx_points = []

        for _, row in df_plot.iterrows():

            alerts = str(row['hdx_alert_level']).lower()

            if 'high concern' in alerts:
                hdx_points.append({
                    'month': row['month'],
                    'level': 'High concern',
                    'color': 'red',
                    'y': 1.08
                })

            if 'medium concern' in alerts:
                hdx_points.append({
                    'month': row['month'],
                    'level': 'Medium concern',
                    'color': 'gold',
                    'y': 1.03
                })

        hdx_alerts = pd.DataFrame(hdx_points)

        if not hdx_alerts.empty:

            fig.add_trace(go.Scatter(
                x=hdx_alerts['month'],
                y=hdx_alerts['y'],

                mode='markers',

                name='HDX Alerts',

                marker=dict(
                    size=7,
                    color=hdx_alerts['color'],
                    symbol='circle',
                    line=dict(width=0.5, color='black')
                ),

                yaxis='y3',

                customdata=hdx_alerts['level'],

                hovertemplate=
                    '<b>HDX SIGNAL</b><br>' +
                    'Date: %{x|%b %Y}<br>' +
                    'Level: %{customdata}<extra></extra>'
            ))

    # --- 4. CONFIGURACIÓN DEL LAYOUT MULTI-EJE ---
    fig.update_layout(
        title=dict(
            text=f'<b>Full Crisis Timeline Overview: {country_code}</b>',
            font=dict(size=22),
            x=0.05
        ),
        margin=dict(l=60, r=100, t=110, b=60), 
        height=600,
        plot_bgcolor='white',
        hovermode="closest",
        
        xaxis=dict(
            domain=[0, 0.85], # Recortamos un poco para que quepa el 3r eje a la derecha
            title=dict(text="<b>Date</b>"),
            showgrid=False,
            dtick="M3",
            tickformat="%b %Y",
            tickangle=45
        ),
        
        # Eje Y1: Izquierda (Fatalidades)
        yaxis=dict(
            title=dict(text="<b>Fatalities</b>", font=dict(color="orange")),
            tickfont=dict(color="orange"),
            showgrid=True,
            gridcolor='lightgrey',
            rangemode="tozero"
        ),
        
        # Eje Y2: Derecha Interna (Desplazamientos Log)
        yaxis2=dict(
            title=dict(text="<b>log(1 + Displacements)</b>", font=dict(color="purple")),
            tickfont=dict(color="purple"),
            anchor="x",
            overlaying="y",
            side="right",
            showgrid=False,
            rangemode="tozero"
        ),
        
        # Eje Y3: Derecha Externa (Risk Score)
        yaxis3=dict(
            title=dict(text="<b>Risk</b>", font=dict(color="black")),
            tickfont=dict(color="black"),
            anchor="free",
            overlaying="y",
            side="right",
            position=1.0,
            range=[0, 1.15],
            showgrid=False
        ),
        
        legend=dict(
            orientation="h",
            yanchor="bottom", y=1.02,
            xanchor="center", x=0.5,
            bgcolor='rgba(255,255,255,0.8)',
            bordercolor='lightgrey',
            borderwidth=1
        )


    )

    fig.show()

# --- EJECUCIÓN ---
plot_full_crisis_timeline(df, cerf_path="../data_clean/cerf_clean.csv", country_code="SYR")